# Prompting Flan-T5

In [ ]:
%pip install --no-deps --ignore-installed -r ../requirements.txt -v

In [ ]:
import transformers
print(transformers.__version__)
import torch
print(torch.__version__)


from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

## Summarisation

In [ ]:
# Define the prompt
text_to_summarize = "Studies show that eating carrots helps improve vision. Carrots contain beta-carotene, a substance that the body converts into vitamin A, crucial for maintaining healthy eyesight."

if not text_to_summarize.strip():
    translated_text = ""
    print("Input is empty. Skipping summarization.")
else:
    summarize_prompt = f'summarize: {text_to_summarize}'

    # Generate inputs
    # max_length = max number of tokens the tokenizer will generate for the given prompt
    inputs = tokenizer(summarize_prompt, return_tensors="pt", max_length=512, truncation=True, padding=True)

    # Generate outputs

    # num_beams
    # This activates Beam Search, a technique that prevents the model from being short-sighted.
    # Instead of just picking the single most likely next word at each step (greedy search), 
    # the model keeps track of the top 5 most promising pathways (beams) at the same time.
    # It calculates the overall probability of entire sentences, 
    # resulting in significantly higher-quality, more coherent, and grammatically correct text.
    # Trade-off: Setting this to 5 means the model does more math background work,
    # making generation roughly 5 times slower than standard greedy search.

    # early_stopping = True
    # The generation structure shuts down the moment all 5 active pathways find an EOS token.
    # This prevents the model from wasting time generating useless padding tokens once the sentence
    # is naturally finished.
    outputs = model.generate(inputs["input_ids"], max_length=150, num_beams=5, early_stopping=True)

    # Decode and print the summary
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f'Summary: {summary}')


## Translation

In [ ]:
# Define the translation prompt
text_to_translate = 'Cheese is delicious.'
translation_prompt = f"translate English to Spanish: {text_to_translate}"

if not text_to_translate.strip():
    translated_text = ""
    print("Input is empty. Skipping translation.")
else:
    # Prepare inputs and generate outputs
    translation_inputs = tokenizer(translation_prompt, return_tensors="pt", max_length=512, truncation=True, padding=True)
    translation_outputs = model.generate(translation_inputs["input_ids"], max_length=40, num_beams=5, early_stopping=True)

    # Decode and display the translation
    translated_text = tokenizer.decode(translation_outputs[0], skip_special_tokens=True)
    print(translated_text)

## Q&A

In [ ]:
# Define the context and question
context_question = "The Great Wall of China is over 13,000 miles long. question: How long is the Great Wall of China?"

# Generate inputs
question_inputs = tokenizer(context_question, return_tensors="pt", max_length=512, truncation=True, padding=True)

# Generate outputs
question_outputs = model.generate(question_inputs["input_ids"], max_length=50, num_beams=5, early_stopping=True)

# Decode and print the answer
print(tokenizer.decode(question_outputs[0], skip_special_tokens=True))
